# Day 1: Snowflake 基本操作

| # | セクション | 学ぶこと | 時間 |
|---|-----------|----------|------|
| 1 | 仮想ウェアハウス | 作成・起動・スケーリング | 10分 |
| 2 | クエリ結果キャッシュ | 自動キャッシュによる高速化 | 10分 |
| 3 | データ変換 | ゼロコピークローン、VARIANT型、SWAP | 15分 |
| 4 | データ復旧 | Time Travel / UNDROP | 10分 |
| 5 | コスト管理 | リソースモニター & バジェット | 10分 |

---

> **前提条件**: `setup.sql` が実行済みであること

## 事前準備：セッションコンテキストの設定

In [ ]:
USE DATABASE tb_101;
USE ROLE accountadmin;

---
## 1. 仮想ウェアハウス

Snowflakeのウェアハウスは**コンピュートリソース**です。

| 特徴 | 説明 |
|------|------|
| 即時スケーリング | XSmall〜6XLargeまで数秒で変更可能 |
| 自動サスペンド/リジューム | 使わない間はコストゼロ |
| コンピュートとストレージの分離 | 複数WHが同じデータに同時アクセス可能 |

### 主要パラメータ

| パラメータ | 説明 | デフォルト |
|-----------|------|----------|
| `WAREHOUSE_SIZE` | コンピューティングリソース量 | XSmall |
| `AUTO_SUSPEND` | 非アクティブ後に自動停止するまでの秒数 | 600 |
| `AUTO_RESUME` | クエリ実行時に自動で再開するか | TRUE |
| `INITIALLY_SUSPENDED` | 作成直後はサスペンド状態にするか | FALSE |

### Step 1: 既存ウェアハウスの確認

In [ ]:
SHOW WAREHOUSES;

### Step 2: ウェアハウスの作成

XSmallサイズ、サスペンド状態で作成します。`AUTO_RESUME = false` にしているので、明示的に起動が必要です。

In [ ]:
CREATE OR REPLACE WAREHOUSE my_wh
    COMMENT = 'My TastyBytes warehouse'
    WAREHOUSE_TYPE = 'standard'
    WAREHOUSE_SIZE = 'xsmall'
    MIN_CLUSTER_COUNT = 1
    MAX_CLUSTER_COUNT = 2
    SCALING_POLICY = 'standard'
    AUTO_SUSPEND = 60
    INITIALLY_SUSPENDED = true
    AUTO_RESUME = false;

### Step 3: サスペンド中のクエリ実行（エラー体験）

ウェアハウスを使用設定し、クエリを実行してみます。

> ⚠️ **サスペンド中のためエラーになります**（意図的です）

In [ ]:
USE WAREHOUSE my_wh;

SELECT
    o.truck_brand_name,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(o.price) AS total_sales
FROM analytics.orders_v o
GROUP BY o.truck_brand_name
ORDER BY total_sales DESC;

### Step 4: ウェアハウスの明示的な起動

`ALTER WAREHOUSE ... RESUME` で起動し、クエリが正常に実行できることを確認します。

In [ ]:
ALTER WAREHOUSE my_wh RESUME;

SELECT
    o.truck_brand_name,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(o.price) AS total_sales
FROM analytics.orders_v o
GROUP BY o.truck_brand_name
ORDER BY total_sales DESC;

### Step 5: スケールアップ

Snowflakeでは実行中でも **即座にサイズ変更** が可能です。ダウンタイムはありません。

In [ ]:
ALTER WAREHOUSE my_wh SET warehouse_size = 'Medium';

SELECT
    DATE(o.order_ts) AS order_date,
    o.truck_brand_name,
    o.primary_city,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(o.price) AS total_sales
FROM harmonized.orders_v o
GROUP BY ALL
ORDER BY total_sales DESC
LIMIT 1000;

---
## 2. クエリ結果キャッシュ

Snowflakeは同じクエリ結果を **24時間自動キャッシュ** します。

| ポイント | 説明 |
|---------|------|
| コンピュート不要 | キャッシュヒット時はWH起動なし |
| 自動無効化 | 元データが変更されると自動で無効化 |
| グローバル | 同じアカウント内の全ユーザーで共有 |

上の売上集計クエリをもう一度実行してください。

**実行時間が劇的に短縮される**ことを確認できます（数秒 → ミリ秒）。

In [ ]:
-- 同じクエリを再実行（キャッシュヒット → 即時返却）
SELECT
    o.truck_brand_name,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(o.price) AS total_sales
FROM analytics.orders_v o
GROUP BY o.truck_brand_name
ORDER BY total_sales DESC;

### スケールダウン

次のセクションではより小さいデータを扱うため、コスト節約のためにXSmallに戻します。

In [ ]:
ALTER WAREHOUSE my_wh SET warehouse_size = 'XSmall';

---
## 3. データ変換：ゼロコピークローン & VARIANT型

| 機能 | 効果 |
|------|------|
| **ゼロコピークローン** | 追加ストレージなしでテーブルの完全コピーを瞬時に作成 |
| **VARIANT型** | JSON等の半構造化データをそのまま格納・クエリ可能 |
| **SWAP WITH** | 開発テーブルと本番テーブルをアトミックに入れ替え |

### シナリオ

`truck_details` テーブルの `truck_build` カラム（VARIANT型）にJSON形式で年式・メーカー・モデルが格納されています。
これを個別カラムに分離する作業を、安全な開発環境で行います。

### Step 1: VARIANT型データの確認

In [ ]:
SELECT truck_id, truck_build
FROM raw_pos.truck_details
ORDER BY truck_id
LIMIT 10;

### Step 2: ゼロコピークローンで開発環境を作成

`CLONE` はメタデータのみの操作で **瞬時に完了** します。開発環境の準備に何時間もかける必要はありません。

In [ ]:
CREATE OR REPLACE TABLE raw_pos.truck_dev CLONE raw_pos.truck_details;

SELECT TOP 15 *
FROM raw_pos.truck_dev
ORDER BY truck_id;

### Step 3: カラム追加 & VARIANTからのデータ抽出

コロン（`:`）演算子でJSONのキーにアクセスし、`::型` でキャストします。

In [ ]:
ALTER TABLE raw_pos.truck_dev ADD COLUMN IF NOT EXISTS year NUMBER;
ALTER TABLE raw_pos.truck_dev ADD COLUMN IF NOT EXISTS make VARCHAR(255);
ALTER TABLE raw_pos.truck_dev ADD COLUMN IF NOT EXISTS model VARCHAR(255);

UPDATE raw_pos.truck_dev
SET
    year = truck_build:year::NUMBER,
    make = truck_build:make::VARCHAR,
    model = truck_build:model::VARCHAR;

In [ ]:
SELECT truck_id, year, make, model
FROM raw_pos.truck_dev
ORDER BY truck_id
LIMIT 10;

### Step 4: データ品質の確認と修正

メーカー名の分布を確認すると、`Ford` と `Ford_` が別カウントになっている問題が見つかります。

In [ ]:
SELECT
    make,
    COUNT(*) AS count
FROM raw_pos.truck_dev
GROUP BY make
ORDER BY make ASC;

In [ ]:
UPDATE raw_pos.truck_dev
    SET make = 'Ford'
    WHERE make = 'Ford_';

SELECT make, COUNT(*) AS count
FROM raw_pos.truck_dev
GROUP BY make
ORDER BY count DESC;

### Step 5: SWAP WITH で本番に昇格

> `ALTER TABLE ... SWAP WITH` はメタデータのみの操作で **瞬時に完了** します。
> Blue/Green デプロイメントのようにダウンタイムゼロで本番切替できます。

In [ ]:
ALTER TABLE raw_pos.truck_details SWAP WITH raw_pos.truck_dev;

SELECT make, COUNT(*) AS count
FROM raw_pos.truck_details
GROUP BY make
ORDER BY count DESC;

### Step 6: クリーンアップ

不要になった `truck_build` カラムを削除し、開発テーブルも削除します。

> ⚠️ **ここでは意図的に間違ったテーブルをDROPします**（次のセクションのため）

In [ ]:
ALTER TABLE raw_pos.truck_details DROP COLUMN truck_build;

-- 間違えて本番テーブルをDROPしてしまった！
DROP TABLE raw_pos.truck_details;

---
## 4. データ復旧：Time Travel & UNDROP

Snowflakeの **Time Travel** 機能により：
- 削除されたオブジェクトを `UNDROP` で即座に復旧
- 過去の任意時点のデータに `AT(TIMESTAMP => ...)` でアクセス
- 保持期間：Standard版 = 1日、Enterprise版 = 最大90日

> さっき本番テーブルを誤って削除してしまいました...
> でも大丈夫。`UNDROP` で一瞬で復旧できます。

In [ ]:
-- テーブルが存在しないことを確認（エラーになる）
DESCRIBE TABLE raw_pos.truck_details;

In [ ]:
-- UNDROP で復元！
UNDROP TABLE raw_pos.truck_details;

-- 復旧を確認
SELECT * FROM raw_pos.truck_details LIMIT 5;

In [ ]:
-- 今度こそ正しく開発テーブルを削除
DROP TABLE IF EXISTS raw_pos.truck_dev;

---
## 5. コスト管理：リソースモニター & バジェット

| 機能 | 対象 | アクション |
|------|------|----------|
| **リソースモニター** | ウェアハウスのクレジット使用量 | 通知 / サスペンド / 即時停止 |
| **バジェット** | あらゆるSnowflakeオブジェクト（ドルベース） | 通知 |

### リソースモニターのアクション
- `NOTIFY` — メール通知を送信
- `SUSPEND` — 実行中クエリの完了後にサスペンド
- `SUSPEND_IMMEDIATE` — 実行中クエリも含め即時停止

In [ ]:
USE ROLE accountadmin;

CREATE OR REPLACE RESOURCE MONITOR my_resource_monitor
    WITH CREDIT_QUOTA = 100
    FREQUENCY = MONTHLY
    START_TIMESTAMP = IMMEDIATELY
    TRIGGERS ON 75 PERCENT DO NOTIFY
             ON 90 PERCENT DO SUSPEND
             ON 100 PERCENT DO SUSPEND_IMMEDIATE;

ALTER WAREHOUSE my_wh
    SET RESOURCE_MONITOR = my_resource_monitor;

### バジェットの作成

バジェットはリソースモニターより柔軟で、ウェアハウス以外のオブジェクト（サーバーレス機能等）もドルベースで管理できます。

> バジェットの設定（上限額、通知先メール、監視対象リソースの追加）は
> Snowsight UI の **Admin → Cost Management → Budgets** から行います。

In [ ]:
CREATE OR REPLACE SNOWFLAKE.CORE.BUDGET my_budget()
    COMMENT = 'My Tasty Bytes Budget';

---
## まとめ

| 学んだこと | Snowflakeの差別化ポイント |
|-----------|-------------------------|
| 仮想ウェアハウス | 即時スケーリング、自動サスペンド、コンピュートとストレージの完全分離 |
| クエリ結果キャッシュ | コンピュート不要で即時返却、グローバル共有 |
| ゼロコピークローン | 瞬時作成、追加ストレージなし、安全な開発環境 |
| Time Travel / UNDROP | 運用レスのデータ復旧、最大90日間 |
| リソースモニター / バジェット | きめ細やかなコスト制御、自動アクション |

---
## リセット（環境のクリーンアップ）

ハンズオン終了後に環境を元に戻す場合は、以下を実行してください。

In [ ]:
USE ROLE accountadmin;

DROP WAREHOUSE IF EXISTS my_wh;
DROP RESOURCE MONITOR IF EXISTS my_resource_monitor;
DROP SNOWFLAKE.CORE.BUDGET IF EXISTS my_budget;
DROP TABLE IF EXISTS raw_pos.truck_dev;

ALTER WAREHOUSE tb_dev_wh SUSPEND;